# **Scraping Planos Supranet**

_Página do provedor: https://supranet.com.br/_

In [1]:
# Bibliotecas
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from datetime import date
import re

## **1. Extração**

In [2]:
# Obter lista de planos
texto_html = open("../../data/scraped/supranet.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
planos = soup.find_all("div", role="group")
print(f"{len(planos)} cartões de planos capturados.")

11 cartões de planos capturados.


In [3]:
# Extrair informações dos planos
def raspar_plano(plano):
    velocidade = plano.find_all("div", attrs={"data-e-type": "widget"})[2].text
    preco = plano.find_all("div", class_="jet-listing-dynamic-field__content")[0].text
    svas = plano.find_all("p")[1].text

    return {"velocidade": velocidade,
            "preco": preco,
            "svas": svas}

dados = []

for i in planos:
    dados_plano = raspar_plano(i)
    dados.append(dados_plano)

df = pd.DataFrame(dados)
df

,velocidade,preco,svas
0,\n\t\t\t\t\t\t\t\t\t700Mega\t\t\t\t\t\t\t\t,"R$ 129,90","* Escolha entre Disney+, HBO Max e GloboPlay. ..."
1,\n\t\t\t\t\t\t\t\t\t800Mega\t\t\t\t\t\t\t\t,"R$ 144,90",/mês
2,\n\t\t\t\t\t\t\t\t\t1000Mega\t\t\t\t\t\t\t\t,"R$ 159,90",/mês
3,\n\t\t\t\t\t\t\t\t\t500Mega\t\t\t\t\t\t\t\t,"R$ 99,90",/mês
4,\n\t\t\t\t\t\t\t\t\t700Mega\t\t\t\t\t\t\t\t,"R$ 109,90",/mês
5,\n\t\t\t\t\t\t\t\t\t700Mega\t\t\t\t\t\t\t\t,"R$ 129,90","* Escolha entre Disney+, HBO Max e GloboPlay. ..."
6,\n\t\t\t\t\t\t\t\t\t800Mega\t\t\t\t\t\t\t\t,"R$ 144,90",/mês
7,\n\t\t\t\t\t\t\t\t\t1000Mega\t\t\t\t\t\t\t\t,"R$ 159,90",/mês
8,\n\t\t\t\t\t\t\t\t\t500Mega\t\t\t\t\t\t\t\t,"R$ 99,90",/mês
9,\n\t\t\t\t\t\t\t\t\t700Mega\t\t\t\t\t\t\t\t,"R$ 109,90",/mês


## **2. Tratamento**

In [4]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Supranet"

# Velocidade
df_tratado["VELOCIDADE"] = df_tratado["velocidade"].str.extract(r"(\d+)", expand=False).astype(float)

# PRECO
df_tratado["PRECO"] = (
    df_tratado["preco"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

# SVAs
df_tratado["SVAs"] = df_tratado["svas"].str.replace(" e ", ", ").str.replace(".", ", ")
df_tratado["SVAs"] = df_tratado["SVAs"].str.replace(", ", " ou ")

df_tratado["SVAs"] = np.where(df_tratado["SVAs"]=="/mês",
                              np.nan,
                              df_tratado["SVAs"])

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

# Remover duplicados
df_tratado = df_tratado.drop_duplicates()

df_tratado

,DATA,ISP,VELOCIDADE,SVAs,PRECO
0,2026-05-12,Supranet,700.0,* Escolha entre Disney+ ou HBO Max ou GloboPla...,129.9
1,2026-05-12,Supranet,800.0,NaN,144.9
2,2026-05-12,Supranet,1000.0,NaN,159.9
3,2026-05-12,Supranet,500.0,NaN,99.9
4,2026-05-12,Supranet,700.0,NaN,109.9


## **3. Carga**

In [5]:
# Salvando
provedor = "SUPRANET.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")

🟢 2026-05-12-SUPRANET.csv salvo!
